# Mechanistic within-fixture bonus — scoping (render-not-decide)

**This notebook DECIDES NOTHING.** It runs, reports, and visualizes the four pre-registered measurements
(Q1–Q4) behind the question *"should bonus be modelled mechanistically — compute each appearing player's
BPS from their contributions, rank all appearances in the fixture, assign 3/2/1 — instead of the incumbent
per-player OLS map `returns_pts → E[bonus]`?"* The pre-registered rule and the **verdict** live in
[docs/model-redesign-bonus-mechanistic-scoping.md](../../../docs/model-redesign-bonus-mechanistic-scoping.md).

Bonus is competitive across **every appearing player in the fixture** (both teams, starters + subs who got
on). The incumbent (`model/terms/bonus/bonus.py`) ignores the fixture entirely and maps a player's *own*
returns to their *own* bonus. Every figure is theme-agnostic and powered-n annotated.

## Setup — scored population, fixture field, and shared walk-forward helpers

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

from dal.pipeline import load as load_mart
from dal.pipeline import load_fixture_map
from model.eval.metrics import block_bootstrap_ci, cell_spearman, has_rank_signal, position_bias
from model.eval.walkforward import MIN_ROWS_PER_POS, POSITIONS, WARMUP_GW
from model.terms.bonus.bonus import BonusModel, returns_points

CONTRIB = ["goals_scored", "assists", "clean_sheets", "saves", "defensive_contribution"]

# --- theme-agnostic styling: transparent surface + neutral ink legible on light AND dark ---
INK = "#6b7280"
POS_COLOR = {"GK": "#0072B2", "DEF": "#009E73", "MID": "#E69F00", "FWD": "#D55E00"}  # Okabe-Ito, CVD-safe
BONUS_COLOR = {0: "#b8c0c9", 1: "#7fb3d5", 2: "#2e86c1", 3: "#1a5276"}  # sequential: how much bonus
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.titlesize": 11})


def style_ax(ax):
    ax.set_facecolor("none")
    for sp in ax.spines.values():
        sp.set_color(INK); sp.set_linewidth(0.8)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.tick_params(colors=INK, labelsize=9)
    for lab in (ax.xaxis.label, ax.yaxis.label, ax.title):
        lab.set_color(INK)
    ax.grid(True, color=INK, alpha=0.15, linewidth=0.6)
    return ax


def new_fig(*a, **k):
    fig, ax = plt.subplots(*a, **k)
    fig.patch.set_alpha(0)
    return fig, ax

In [ ]:
# Build the scored frame: incumbent population (minutes>0, non-DGW) + physical-fixture field + features.
def build_scoped_frame():
    fmap = load_fixture_map()  # sanctioned (player_id, gw, fixture_id) key for SGW rows — via dal.pipeline

    mart = load_mart().mart
    df = mart[(mart["minutes"] > 0) & (~mart["is_dgw"].astype(bool))].copy()
    df["returns_pts"] = returns_points(df)
    df["bonus"] = pd.to_numeric(df["bonus"], errors="coerce")
    df["bps"] = pd.to_numeric(df["bps"], errors="coerce")
    for c in CONTRIB:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    df = df.merge(fmap, on=["player_id", "gw"], how="inner").dropna(subset=["fixture_id", "bonus", "bps"])
    df = df[df.groupby("fixture_id")["bonus"].transform("sum") >= 6].copy()  # fully-awarded fixtures only

    def n_ahead(s):  # competitive count: appearances in the fixture with strictly higher score
        v = s.to_numpy(float)
        return pd.Series((v[:, None] < v[None, :]).sum(1), index=s.index)

    df["n_ahead_rp"] = df.groupby("fixture_id")["returns_pts"].apply(n_ahead).reset_index(level=0, drop=True)
    third = df.groupby("fixture_id")["returns_pts"].transform(
        lambda s: np.sort(s.to_numpy(float))[::-1][min(2, len(s) - 1)])
    df["gap_to_3rd_rp"] = df["returns_pts"] - third
    return df.reset_index(drop=True)


def wf_predict(df, feats, target, clip=None):
    """Expanding walk-forward (gw<t) per-position OLS prediction, aligned to df.index."""
    pred = pd.Series(np.nan, index=df.index, dtype=float)
    for pos in POSITIONS:
        sdf = df[df["position"] == pos]
        for t in sorted(g for g in sdf["gw"].unique() if g > WARMUP_GW):
            tr = sdf[sdf["gw"] < t].dropna(subset=[*feats, target])
            te = sdf[sdf["gw"] == t]
            if len(tr) < 50 or te.empty:
                continue
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                res = sm.OLS(tr[target].to_numpy(float),
                             sm.add_constant(tr[feats].to_numpy(float), has_constant="add")).fit()
                p = res.predict(sm.add_constant(te[feats].to_numpy(float), has_constant="add"))
            pred.loc[te.index] = np.clip(p, *clip) if clip else p
    return pred


def assign_bonus(df, score_col):
    """Within-fixture FPL bonus assignment from a ranking score (higher=better): bonus = f(n_ahead)."""
    def na(s):
        v = s.to_numpy(float)
        return pd.Series((v[:, None] < v[None, :]).sum(1), index=s.index)
    ahead = df.groupby("fixture_id")[score_col].apply(na).reset_index(level=0, drop=True)
    return ahead.map({0: 3.0, 1: 2.0, 2: 1.0}).fillna(0.0)


df = build_scoped_frame()
print(f"scored rows: {len(df):,}   fixtures: {df['fixture_id'].nunique()}   gw {df.gw.min()}-{df.gw.max()}")
display(df.groupby("position").size().reindex(list(POSITIONS)).rename("rows").to_frame().T)

## Q1 — CEILING: how much bonus does the incumbent per-player map already capture?

The floor any rewrite must beat: within-`(gw, position)` `Spearman(e_bonus, realized bonus)` and the level
(`position_bias`). Figure 1 shows the incumbent calibration — E[bonus] vs realized bonus by `returns_pts`.

In [ ]:
inc = BonusModel().fit(load_mart().mart)
pop = BonusModel.population(load_mart().mart).assign(e_bonus=inc.predictions)
df = df.merge(pop[["player_id", "gw", "e_bonus"]], on=["player_id", "gw"], how="left")
ev1 = df[(df.gw > WARMUP_GW) & df.e_bonus.notna()].copy()

q1 = []
for pos in POSITIONS:
    s = ev1[ev1.position == pos]
    rhos = [cell_spearman(g.e_bonus.to_numpy(), g.bonus.to_numpy())
            for _, g in s.groupby("gw") if has_rank_signal(g, "e_bonus", "bonus", MIN_ROWS_PER_POS)]
    q1.append({"position": pos, "spearman_ceiling": round(float(np.mean(rhos)), 3), "n_gw": len(rhos), "n": len(s)})
q1 = pd.DataFrame(q1)
print("Q1 within-(gw,pos) Spearman(e_bonus, bonus) — the ceiling:")
display(q1)
print("Level (position_bias):")
display(position_bias(ev1, "e_bonus", "bonus")[["position", "n", "mean_pred", "mean_target", "bias", "rel_bias", "ok"]])

In [ ]:
# Figure 1 — incumbent calibration: E[bonus] vs realized bonus, per position, by returns_pts level.
fig, axes = new_fig(1, 4, figsize=(13.5, 3.2), sharey=True)
for ax, pos in zip(axes, POSITIONS):
    s = ev1[ev1.position == pos].copy()
    s["rp"] = s.returns_pts.round().clip(upper=12)
    g = s.groupby("rp").agg(exp=("e_bonus", "mean"), real=("bonus", "mean"), n=("bonus", "size"))
    ax.plot(g.index, g.real, "-o", color=POS_COLOR[pos], lw=2, ms=5, label="realized")
    ax.plot(g.index, g.exp, "--", color=INK, lw=1.6, label="E[bonus] (incumbent)")
    ax.set_title(f"{pos}  (rho={q1.loc[q1.position==pos,'spearman_ceiling'].item():.2f}, n={len(s):,})")
    ax.set_xlabel("returns_pts"); style_ax(ax)
axes[0].set_ylabel("bonus")
axes[0].legend(frameon=False, fontsize=8, labelcolor=INK)
fig.suptitle("Q1 — Incumbent calibration: E[bonus] tracks realized bonus by own returns_pts", color=INK, y=1.04)
plt.tight_layout(); plt.show()

## Q2 — COMPETITIVE RESIDUAL: does conditioning on the fixture beat own returns?

Two walk-forward per-position OLS predictors of realized bonus: **(a)** `returns_pts` alone (incumbent) vs
**(b)** `returns_pts` + within-fixture competitive features (`n_ahead`, `gap_to_3rd`, both from *modelled*
returns — deployable). Metric: paired per-`(gw, position)` `Spearman(pred, bonus)` **delta (b − a)**,
block-bootstrapped over the per-GW series. **A real, useful competitive signal needs the CI above 0.**

Figure 2 first shows the *mechanism* — for a sample of fixtures, every appearing player at
`(returns_pts, bps)`, colored by the bonus they won: the top-3 **BPS** (not top returns) take 3/2/1, and
two hauling teammates split the pot.

In [ ]:
# Figure 2a — the competition: sample fixtures, all appearing players, who wins the 3/2/1.
cand = (df.assign(hi=df.returns_pts >= 6).groupby("fixture_id")
        .agg(nhi=("hi", "sum"), n=("bonus", "size")).query("nhi >= 3 and n >= 20"))
sample = cand.sort_values("nhi", ascending=False).head(6).index.tolist()
fig, axes = new_fig(2, 3, figsize=(13.5, 6.4))
for ax, fx in zip(axes.ravel(), sample):
    g = df[df.fixture_id == fx]
    for tier in [0, 1, 2, 3]:
        gt = g[g.bonus == tier]
        ax.scatter(gt.returns_pts, gt.bps, s=44 if tier else 26, color=BONUS_COLOR[tier],
                   edgecolor="white", linewidth=0.6, label=f"bonus {tier}" if tier else "no bonus", zorder=3 if tier else 2)
    for _, r in g[g.bonus > 0].iterrows():
        ax.annotate(f"{int(r.bonus)}", (r.returns_pts, r.bps), fontsize=8, color=INK,
                    xytext=(3, 3), textcoords="offset points")
    ax.set_title(f"fixture {int(fx)}  (n={len(g)})"); ax.set_xlabel("returns_pts"); ax.set_ylabel("bps"); style_ax(ax)
axes.ravel()[0].legend(frameon=False, fontsize=7.5, labelcolor=INK, loc="upper left")
fig.suptitle("Q2 — Within-fixture competition: top-3 BPS take 3/2/1 (both teams, hauling teammates split)",
             color=INK, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# Q2 — paired within-(gw,pos) Spearman delta (competitive OLS minus incumbent OLS) + block-bootstrap CI.
df["pred_a"] = wf_predict(df, ["returns_pts"], "bonus", clip=(0.0, 3.0))
df["pred_b"] = wf_predict(df, ["returns_pts", "n_ahead_rp", "gap_to_3rd_rp"], "bonus", clip=(0.0, 3.0))
ev2 = df[(df.gw > WARMUP_GW) & df.pred_a.notna() & df.pred_b.notna()].copy()

q2 = []
for pos in POSITIONS:
    s = ev2[ev2.position == pos]
    deltas = []
    for _, g in s.groupby("gw"):
        if has_rank_signal(g, "pred_a", "bonus", MIN_ROWS_PER_POS) and has_rank_signal(g, "pred_b", "bonus", MIN_ROWS_PER_POS):
            deltas.append(cell_spearman(g.pred_b.to_numpy(), g.bonus.to_numpy())
                          - cell_spearman(g.pred_a.to_numpy(), g.bonus.to_numpy()))
    deltas = np.asarray(deltas)
    lo, hi = block_bootstrap_ci(deltas)
    q2.append({"position": pos, "delta_b_minus_a": round(float(deltas.mean()), 4),
               "ci_lo": round(lo, 4), "ci_hi": round(hi, 4), "n_gw": len(deltas),
               "rows_per_cell": round(len(s) / max(len(deltas), 1), 1),
               "excludes_0": bool(np.isfinite(lo) and (lo > 0 or hi < 0)),
               "improves": bool(np.isfinite(lo) and lo > 0)})
q2 = pd.DataFrame(q2)
display(q2)

In [ ]:
# Figure 2b — the competitive delta with block-bootstrap 95% CI. Above 0 = fixture info helps.
fig, ax = new_fig(figsize=(8.4, 3.6))
y = np.arange(len(q2))[::-1]
ax.axvline(0, color=INK, lw=1.2, zorder=1)
for yi, (_, r) in zip(y, q2.iterrows()):
    c = POS_COLOR[r.position]
    ax.plot([r.ci_lo, r.ci_hi], [yi, yi], color=c, lw=3, solid_capstyle="round", zorder=2)
    ax.scatter([r.delta_b_minus_a], [yi], color=c, s=70, edgecolor="white", linewidth=1, zorder=3)
    ax.annotate(f"n_gw={r.n_gw}, ~{r.rows_per_cell:.0f}/cell", (r.ci_hi, yi), fontsize=8, color=INK,
                xytext=(6, 0), textcoords="offset points", va="center")
ax.set_yticks(y); ax.set_yticklabels(q2.position)
ax.set_xlabel("paired within-(gw,pos) Spearman delta  (competitive − incumbent)")
ax.set_title("Q2 — Adding fixture-competitive returns features: delta vs 0 (95% block-bootstrap CI)")
style_ax(ax); plt.tight_layout(); plt.show()

## Q3 — BPS RECONSTRUCTION: how much BPS can the modelled contributions see?

Per position, regress realized `bps` on the modelled contributions
`[goals, assists, clean_sheets, saves, defensive_contribution]`. `R²` / Spearman size how much BPS is
reconstructable; the **residual** (passing / tackles / CBI / recoveries / cards) is the error a
mechanistic allocator inherits. The **tracking** bar asks the decisive question: does ranking a fixture by
*modelled* BPS recover the actual 3/2/1 recipients better than plain `returns_pts`?

In [ ]:
q3, resid_by_pos = [], {}
for pos in POSITIONS:
    s = df[df.position == pos].dropna(subset=[*CONTRIB, "bps"])
    res = sm.OLS(s.bps.to_numpy(float), sm.add_constant(s[CONTRIB].to_numpy(float), has_constant="add")).fit()
    fit = res.predict(sm.add_constant(s[CONTRIB].to_numpy(float), has_constant="add"))
    resid = s.bps.to_numpy(float) - fit
    resid_by_pos[pos] = resid
    q3.append({"position": pos, "R2": round(res.rsquared, 3), "spearman": round(cell_spearman(fit, s.bps.to_numpy(float)), 3),
               "resid_sd_bps": round(float(resid.std()), 2), "resid_var_bps": round(float(resid.var()), 1), "n": len(s)})
q3 = pd.DataFrame(q3)
display(q3)

# tracking: top-3 recovery of realized bonus recipients, per ranking score (walk-forward modelled bps)
df["fit_bps"] = wf_predict(df, CONTRIB, "bps")
trk = df[df.fit_bps.notna()].copy()
for score, name in [("returns_pts", "returns_pts"), ("fit_bps", "modelled_bps"), ("bps", "oracle_bps")]:
    ah = trk.groupby("fixture_id")[score].apply(
        lambda s: pd.Series((s.to_numpy(float)[:, None] < s.to_numpy(float)[None, :]).sum(1), index=s.index)
    ).reset_index(level=0, drop=True)
    trk[f"top3_{name}"] = ah < 3
rec = trk[trk.bonus > 0]
hit = {n: rec[f"top3_{n}"].mean() for n in ["returns_pts", "modelled_bps", "oracle_bps"]}
print(f"top-3 hit-rate of {len(rec)} realized bonus recipients:", {k: round(v, 3) for k, v in hit.items()})

In [ ]:
# Figure 3 — (left) bps reconstruction residual per position; (right) top-3 recovery per ranking score.
fig, (axL, axR) = new_fig(1, 2, figsize=(13.5, 4.0), gridspec_kw={"width_ratios": [1.4, 1]})
parts = axL.violinplot([resid_by_pos[p] for p in POSITIONS], showextrema=False, showmedians=True)
for b, p in zip(parts["bodies"], POSITIONS):
    b.set_facecolor(POS_COLOR[p]); b.set_alpha(0.55); b.set_edgecolor(INK)
parts["cmedians"].set_color(INK)
axL.axhline(0, color=INK, lw=1)
axL.set_xticks(range(1, 5)); axL.set_xticklabels([f"{p}\nR2={q3.loc[q3.position==p,'R2'].item():.2f}" for p in POSITIONS])
axL.set_ylabel("realized bps − modelled-contribution fit"); axL.set_title("Q3 — Unreconstructable BPS residual (passing/tackles/CBI/cards)")
style_ax(axL)

order = ["returns_pts", "modelled_bps", "oracle_bps"]
bar_c = ["#9aa4ae", "#e69f00", "#1a5276"]
axR.bar(range(3), [hit[n] for n in order], color=bar_c, edgecolor="white", linewidth=1, width=0.66)
axR.axhline(hit["returns_pts"], color=INK, ls="--", lw=1, alpha=0.6)
for i, n in enumerate(order):
    axR.annotate(f"{hit[n]:.2f}", (i, hit[n]), ha="center", va="bottom", fontsize=10, color=INK)
axR.set_xticks(range(3)); axR.set_xticklabels(["returns_pts\n(incumbent)", "modelled\nbps", "oracle\n(realized bps)"])
axR.set_ylim(0, 1.08); axR.set_ylabel("top-3 recovery of bonus recipients")
axR.set_title(f"Q3 — Fixture top-3 recovery (n={len(rec):,})"); style_ax(axR)
plt.tight_layout(); plt.show()

## Q4 — PT-VARIANCE PRIZE: is the recoverable variance material?

`bonus` **is** points, so `Var(bonus | model)` is pt-variance directly. We compare the incumbent residual
`Var(a)` (≈ the ~0.2 pt-var the interval-dispersion doc flagged) against what each candidate actually
**removes**: the deployable competitive OLS (**recov_b**), the mechanistic modelled-BPS allocator
(**recov_mech**), and the **oracle** realized-BPS allocator (**recov_oracle**, the ceiling if BPS were
known). Materiality bar: recoverable > **0.10** at **≥2 positions**.

In [ ]:
df["mech"] = assign_bonus(df, "fit_bps")
df["oracle"] = assign_bonus(df, "bps")
e4 = df[df.pred_a.notna() & df.pred_b.notna() & df.fit_bps.notna()].copy()
q4 = []
for pos in POSITIONS:
    s = e4[e4.position == pos]
    va = float((s.bonus - s.pred_a).var())
    q4.append({"position": pos, "var_bonus": round(float(s.bonus.var()), 3), "var_a_incumbent": round(va, 3),
               "recov_b": round(va - float((s.bonus - s.pred_b).var()), 3),
               "recov_mech": round(va - float((s.bonus - s.mech).var()), 3),
               "recov_oracle": round(va - float((s.bonus - s.oracle).var()), 3), "n": len(s)})
q4 = pd.DataFrame(q4)
display(q4)

In [ ]:
# Figure 4 — recoverable pt-variance per position vs the 0.10 materiality bar and the ~0.2 doc reference.
fig, ax = new_fig(figsize=(9.6, 4.0))
x = np.arange(len(POSITIONS)); w = 0.26
series = [("recov_b", "deployable competitive OLS", "#9aa4ae"),
          ("recov_mech", "mechanistic (modelled BPS)", "#e69f00"),
          ("recov_oracle", "oracle (realized BPS, ceiling)", "#1a5276")]
for i, (col, lab, c) in enumerate(series):
    ax.bar(x + (i - 1) * w, q4[col], width=w, color=c, edgecolor="white", linewidth=0.8, label=lab)
ax.axhline(0.10, color="#c0392b", ls="--", lw=1.4, label="materiality bar (0.10)")
ax.axhline(0.20, color=INK, ls=":", lw=1.2, alpha=0.7, label="~0.2 pt-var residual (doc ref)")
ax.set_xticks(x); ax.set_xticklabels(POSITIONS)
ax.set_ylabel("recoverable pt-variance  (Var(a) − Var(model))")
ax.set_ylim(top=float(q4[["recov_oracle"]].max().item()) * 1.30)
ax.set_title("Q4 — Recoverable bonus pt-variance: only the *oracle* clears the bar")
ax.legend(frameon=False, fontsize=8, labelcolor=INK, ncol=2, loc="upper center")
style_ax(ax); plt.tight_layout(); plt.show()

## Verdict — read it in the docs, not here

This notebook decided nothing. The pre-registered decision rule and the **BUILD/REFUTE verdict with its
CIs** live in
[docs/model-redesign-bonus-mechanistic-scoping.md](../../../docs/model-redesign-bonus-mechanistic-scoping.md).
Summary of what the figures show: the incumbent already captures the bonus ranking (Q1); the fixture-
competitive returns features do **not** improve it (Q2, delta ≤ 0); modelled contributions cannot
reconstruct BPS well enough to out-rank plain `returns_pts` within a fixture (Q3); and the recoverable
pt-variance sits far below the materiality bar unless BPS is known exactly (Q4, only the oracle clears it).